# Pregunta 3 — Estacionalidad y distribución geográfica
Analítica Computacional para la Toma de Decisiones — Proyecto 1 (SENA / SECOP II)

**Pregunta de negocio:** ¿En qué departamentos/regionales y en qué meses del año se
concentra la firma de estos contratos (estacionalidad)? Esto le dice a una empresa de
outsourcing de talento humano dónde y cuándo sería más viable arrancar un piloto o
negociar la propuesta con el SENA.

**Columnas que usamos de `datos_limpios.csv`:**

- `regional`, `departamento`, `ciudad` — distribución geográfica
- `fecha_de_firma`, `anio_firma`, `mes_firma` — estacionalidad temporal
- `duracion_dias`, `duracion_inconsistente` — características de apoyo (tamaño del compromiso)
- `es_contratacion_individual` — filtro al universo de negocio (outsourcing: prestación de
  servicios con personas naturales)
- `id_contrato` — para conteos
- `valor_del_contrato` — **ojo:** esta columna trae valores extremos que no son razonables
  (máximo del orden de 10^16, muy por encima de un contrato real de SENA) — probablemente
  un problema de parseo en un puñado de filas. No la usamos para estadísticas de esta
  pregunta; si la necesitas en otra parte del análisis, revisa/recorta esos outliers antes.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")


## 1. Carga y preparación

In [ ]:
RUTA_CSV = "datos_limpios.csv"  # ajusta la ruta si es necesario

df = pd.read_csv(RUTA_CSV)
df['fecha_de_firma'] = pd.to_datetime(df['fecha_de_firma'], errors='coerce')

print("Filas totales:", df.shape[0])
print("Rango de fechas:", df['fecha_de_firma'].min(), "a", df['fecha_de_firma'].max())


## 2. Filtro al universo de negocio

Nos quedamos con los contratos de prestación de servicios con personas naturales
(`es_contratacion_individual == True`), que es el universo relevante para la empresa de
outsourcing — el mismo filtro usado en las Preguntas 1 y 2, para que las tres respuestas
sean consistentes entre sí.

In [ ]:
dfb = df[df['es_contratacion_individual'] == True].copy()
print(f"Universo de negocio: {dfb.shape[0]} filas ({dfb.shape[0]/df.shape[0]*100:.1f}% del total SENA)")


## 3. Nuevas características para el análisis de estacionalidad

A partir de `mes_firma` y `fecha_de_firma` construimos columnas auxiliares para poder agrupar y graficar la estacionalidad:

- **`nombre_mes`**: traduce el número de mes (1-12) a su abreviatura en español (Ene, Feb, ...), solo para que los ejes de las gráficas se vean legibles.
- **`trimestre`**: agrupa los meses de 3 en 3 (Q1-Q4), útil para comparar duración de contrato por trimestre más adelante.
- **`anio_mes`**: combina año y mes en un periodo (`YYYY-MM`), necesario para graficar la serie de tiempo real mes a mes (no solo el promedio por mes del año).
- **`dfb_dur`**: copia de `dfb` sin las 9 filas con duración inconsistente, para que esas filas no distorsionen los análisis de duración (`duracion_dias`) más adelante.


In [ ]:
nombres_mes = {1:'Ene', 2:'Feb', 3:'Mar', 4:'Abr', 5:'May', 6:'Jun',
               7:'Jul', 8:'Ago', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dic'}

dfb['nombre_mes'] = dfb['mes_firma'].map(nombres_mes)
dfb['trimestre'] = ((dfb['mes_firma'] - 1) // 3 + 1)

# Año-mes como periodo, para la serie de tiempo real (no solo el patrón promedio)
dfb['anio_mes'] = dfb['fecha_de_firma'].dt.to_period('M')

# Excluimos las 9 filas con duración inconsistente solo para los análisis de duración
dfb_dur = dfb[~dfb['duracion_inconsistente']].copy()

dfb[['fecha_de_firma', 'mes_firma', 'nombre_mes', 'trimestre', 'anio_mes']].head()


## 4. Estadísticas descriptivas generales

(dejamos fuera `valor_del_contrato` por el problema de outliers señalado arriba;
usamos `duracion_dias` como variable numérica de apoyo)

In [ ]:
print("Contratos con fecha de firma válida:", dfb['fecha_de_firma'].notna().sum(),
      f"({dfb['fecha_de_firma'].notna().mean()*100:.1f}%)")
print()
print("Departamentos distintos:", dfb['departamento'].nunique())
print("Regionales distintas:", dfb['regional'].nunique())
print("Ciudades distintas:", dfb['ciudad'].nunique())
print()
dfb_dur['duracion_dias'].describe()


## 5. Distribución geográfica — ¿dónde?

Contamos cuántos contratos hay por `regional` y por `ciudad`, y graficamos las 15 con más volumen en cada caso (barras horizontales, ordenadas de menor a mayor):

- **Por regional**: además del conteo, calculamos qué porcentaje representa cada una sobre el total del universo de negocio (`dfb`) — esto nos dice dónde se concentra la mayoría de la contratación.
- **Por ciudad**: mismo tipo de gráfica, pero a nivel de ciudad, para ver si la concentración regional se explica por una sola ciudad grande o está repartida.


In [ ]:
conteo_regional = dfb['regional'].value_counts()
top_regionales = conteo_regional.head(15)

fig, ax = plt.subplots(figsize=(9, 6))
top_regionales.sort_values().plot(kind='barh', ax=ax)
ax.set_xlabel('Número de contratos')
ax.set_title('Contratos de prestación de servicios individuales por regional SENA (Top 15)')
for i, v in enumerate(top_regionales.sort_values()):
    ax.text(v, i, f' {v:,}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('p3_barras_regional.png', dpi=150)
plt.show()

(top_regionales / dfb.shape[0] * 100).round(1)


In [ ]:
conteo_ciudad = dfb['ciudad'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(9, 6))
conteo_ciudad.sort_values().plot(kind='barh', ax=ax)
ax.set_xlabel('Número de contratos')
ax.set_title('Contratos de prestación de servicios individuales por ciudad (Top 15)')
plt.tight_layout()
plt.savefig('p3_barras_ciudad.png', dpi=150)
plt.show()


## 6. Estacionalidad mensual — ¿cuándo?

In [ ]:
orden_meses = [nombres_mes[m] for m in range(1, 13)]
conteo_mes = dfb['nombre_mes'].value_counts().reindex(orden_meses)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(conteo_mes.index, conteo_mes.values, marker='o', linewidth=2)
ax.fill_between(range(len(conteo_mes)), conteo_mes.values, alpha=0.3)
ax.set_ylabel('Número de contratos (acumulado 2016-2026)')
ax.set_title('Estacionalidad mensual de la firma de contratos — SENA')
ax.set_xticks(range(len(orden_meses)))
ax.set_xticklabels(orden_meses)
plt.tight_layout()
plt.savefig('p3_linea_estacionalidad.png', dpi=150)
plt.show()

conteo_mes


In [ ]:
# Serie de tiempo real (no solo el patrón promedio): ¿se repite el patrón año a año,
# o es una tendencia de crecimiento/caída general?
serie_mensual = dfb.groupby('anio_mes').size()

fig, ax = plt.subplots(figsize=(12, 5))
serie_mensual.plot(ax=ax, linewidth=1.5)
ax.set_ylabel('Número de contratos')
ax.set_xlabel('Año-mes')
ax.set_title('Serie de tiempo mensual real de contratos firmados')
plt.tight_layout()
plt.savefig('p3_serie_tiempo.png', dpi=150)
plt.show()


## 7. Heatmap mes x regional

In [ ]:
top10_regionales = conteo_regional.head(10).index
tabla = (
    dfb[dfb['regional'].isin(top10_regionales)]
    .pivot_table(index='regional', columns='nombre_mes', values='id_contrato',
                 aggfunc='count', fill_value=0)
    .reindex(columns=orden_meses)
    .loc[top10_regionales]  # mantener el orden de mayor a menor volumen
)

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(tabla, cmap='Blues', annot=True, fmt='d', linewidths=1,
            cbar_kws={'label': 'Número de contratos'}, ax=ax)
ax.set_title('Contratos por mes y regional (Top 10 regionales)')
ax.set_xlabel('Mes de firma')
ax.set_ylabel('Regional')
plt.tight_layout()
plt.savefig('p3_heatmap_mes_regional.png', dpi=150)
plt.show()


## 8. Diagrama de dispersión — ¿es consistente el patrón estacional año a año?

Un punto por combinación (año, mes): si el patrón estacional es real y estable, los
puntos de un mismo mes deberían agruparse en un rango similar de conteo entre años.

In [ ]:
dispersión = dfb.dropna(subset=['anio_firma', 'mes_firma']).groupby(
    ['anio_firma', 'mes_firma']
).size().reset_index(name='conteo')

norm = mcolors.Normalize(vmin=dispersión['anio_firma'].min(), vmax=dispersión['anio_firma'].max())

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(dispersión['mes_firma'], dispersión['conteo'],
                 c=dispersión['anio_firma'], cmap='viridis', norm=norm,
                 s=60, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Mes de firma')
ax.set_ylabel('Número de contratos ese mes')
ax.set_title('Volumen mensual de contratos por año (color = año)')
ax.set_xticks(range(1, 13))
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Año')
plt.tight_layout()
plt.savefig('p3_dispersion_mes_anio.png', dpi=150)
plt.show()


## 9. Distribución de la duración del contrato (histograma, caja y violín)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(dfb_dur['duracion_dias'], bins=40, edgecolor='white')
ax.set_xlabel('Duración del contrato (días)')
ax.set_ylabel('Frecuencia')
ax.set_title('Histograma de duración de los contratos (universo de outsourcing)')
plt.tight_layout()
plt.savefig('p3_histograma_duracion.png', dpi=150)
plt.show()


In [ ]:
top6_regionales = conteo_regional.head(6).index
dfb_top6 = dfb_dur[dfb_dur['regional'].isin(top6_regionales)]

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=dfb_top6, x='regional', y='duracion_dias', order=top6_regionales, ax=ax)
ax.set_xlabel('Regional')
ax.set_ylabel('Duración del contrato (días)')
ax.set_title('Duración de contrato por regional (Top 6 en volumen)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('p3_boxplot_duracion_regional.png', dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.violinplot(data=dfb_dur, x='trimestre', y='duracion_dias', ax=ax)
ax.set_xlabel('Trimestre de firma')
ax.set_ylabel('Duración del contrato (días)')
ax.set_title('Distribución de la duración del contrato por trimestre de firma')
plt.tight_layout()
plt.savefig('p3_violin_duracion_trimestre.png', dpi=150)
plt.show()


## 10. Pruebas estadísticas

**Chi-cuadrado de independencia (mes x regional):** ¿el patrón estacional (en qué mes se
firma) es el mismo en todas las regionales, o cambia según la regional? Si el resultado es
significativo, significa que la mejor época para negociar/lanzar el piloto **depende de
en qué regional** se quiera empezar — no hay una única ventana de tiempo nacional.

In [ ]:
chi2, p_valor, gl, esperado = stats.chi2_contingency(tabla)
print(f"Chi-cuadrado = {chi2:.1f}, gl = {gl}, p-valor = {p_valor:.2e}")
if p_valor < 0.05:
    print("-> Se rechaza independencia: el patrón mensual SÍ varía significativamente entre regionales.")
else:
    print("-> No hay evidencia de que el patrón mensual varíe entre regionales.")


**Kruskal-Wallis:** ¿la duración típica de los contratos difiere entre regionales? (se usa
una prueba no paramétrica porque la duración no sigue una distribución normal, como se ve
en el histograma).

In [ ]:
grupos = [dfb_top6.loc[dfb_top6['regional'] == r, 'duracion_dias'].dropna() for r in top6_regionales]
h_stat, p_valor_kw = stats.kruskal(*grupos)
print(f"Kruskal-Wallis H = {h_stat:.1f}, p-valor = {p_valor_kw:.2e}")
if p_valor_kw < 0.05:
    print("-> Las regionales SÍ difieren significativamente en la duración típica de sus contratos.")
else:
    print("-> No hay evidencia de diferencias significativas en duración entre regionales.")


## 11. Hallazgos (para completar con los resultados reales al ejecutar)

- Regionales/departamentos donde más se concentra la contratación: _______
- Meses de mayor concentración (posible ventana de negociación/piloto): _______
- ¿El patrón estacional es igual en todas las regionales o difiere (según el chi-cuadrado)?: _______
- ¿La duración típica de contrato varía por regional (según Kruskal-Wallis)? ¿Qué implica
  eso para el tamaño del compromiso que se le pediría al SENA en cada regional?: _______
